In [ ]:
import pandas as pd
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import datetime
from scipy.interpolate import griddata


import cartopy.crs as ccrs

In [ ]:
# Opening the dataset
ds = pd.read_csv('../data/DRIFT_DATA_TRAIN.csv')

ds['time']=pd.to_datetime(ds[['year',	'month',	'day']]) # Creating datetime index
ds=ds.drop(['year','month','day','doy'], axis=1) # removing all other time info
ds=ds.set_index(['time'])
ds=ds.drop_duplicates()# just in case

daily={time: dailydata for time,dailydata in ds.groupby(ds.index.date)} # group by daily data and turn to dict
ntime=len(daily) # Useful for later

# load the bathymetry as a proxy for the land
bath=pd.read_csv('../data/bathymetry_EASE.csv', header= None)

datamask= (1-bath.isna()) # make sure to check where we have data
land=(bath.where(bath==3000,np.nan)-3000)


lat_grid = pd.read_csv('../data/latitude_EASE.csv', header=None).values
lon_grid = pd.read_csv('../data/longitude_EASE.csv', header=None).values






land_arr = land.to_numpy().astype(np.float32)  # compute once

u_slice = np.empty_like(land_arr)
v_slice = np.empty_like(land_arr)
interU = np.tile(land_arr, (ntime, 1, 1))
interV = interU.copy()

y_idx = np.arange(interU.shape[1])
x_idx = np.arange(interU.shape[2])

grid_y, grid_x = np.meshgrid(y_idx, x_idx, indexing='ij')  # once, outside the loop

for i, day in enumerate(daily.keys()):
    u_slice[:] = land_arr  # cheap in-place copy of an already-numpy array
    v_slice[:] = land_arr
    rows = daily[day]['y_EASE'].astype(int).to_numpy()
    cols = daily[day]['x_EASE'].astype(int).to_numpy()
    u_slice[rows, cols] = daily[day]['u_buoy']
    v_slice[rows, cols] = daily[day]['v_buoy']

    valid_u = ~np.isnan(u_slice)
    valid_v = ~np.isnan(v_slice)
    interU[i] = griddata((grid_y[valid_u], grid_x[valid_u]), u_slice[valid_u], (grid_y, grid_x), method='linear')
    interV[i] = griddata((grid_y[valid_v], grid_x[valid_v]), v_slice[valid_v], (grid_y, grid_x), method='linear')
# for i,day in enumerate(daily.keys()):
#     rows = daily[day]['y_EASE'].astype(int).to_numpy()
#     cols = daily[day]['x_EASE'].astype(int).to_numpy()
#     ucube[i,rows,cols]=daily[day]['u_buoy']
#     vcube[i,rows,cols]=daily[day]['v_buoy']



# for i, day in enumerate(daily.keys()):
#     rows = daily[day]['y_EASE'].astype(int).to_numpy()
#     cols = daily[day]['x_EASE'].astype(int).to_numpy()
#     ucube[i, rows, cols] = daily[day]['u_buoy']
#     vcube[i, rows, cols] = daily[day]['v_buoy']

#     valid_u = ~np.isnan(ucube[i])
#     valid_v = ~np.isnan(vcube[i])

#     interU[i] = griddata(
#         (grid_y[valid_u], grid_x[valid_u]),
#         ucube[i][valid_u],
#         (grid_y, grid_x),
#         method='linear'
#     )
#     interV[i] = griddata(
#         (grid_y[valid_v], grid_x[valid_v]),
#         vcube[i][valid_v],
#         (grid_y, grid_x),
#         method='linear'
#     )


ds = xr.Dataset(
    {
        'u_inter':(['time', 'y', 'x'], interU),
        'v_inter':(['time', 'y', 'x'], interV)
    },
    coords={
        'time': list(daily.keys()),
        'y': y_idx,
        'x': x_idx,
        'lat': (['y', 'x'], lat_grid),
        'lon': (['y', 'x'], lon_grid),
    }
)

ds.to_netcdf('./interpolated_velocities.nc','w')

# uarr[rows,cols]=day1['u_buoy'].to_numpy()


# plt.imshow(uarr)
# plt.colorbar()
# plt.show()

# print(ucube.shape)
# # for i in day1:

# #     print(i)
# #     umap[day1[datetime(i)]['y_EASE'].astype(int), day1[i]['x_EASE'].astype(int)]=day1[i]['u_buoy']





# plt.imshow(land)
# plt.colorbar()
# plt.show()
# plt.imshow(datamask)
# plt.colorbar()


# xr.DataArray(
#     np.full_like(lat_grid,np.nan),
    
# )

In [ ]:
# Opening the dataset
ds = pd.read_csv('../data/DRIFT_DATA_TRAIN.csv')

ds['time']=pd.to_datetime(ds[['year',	'month',	'day']]) # Creating datetime index
ds=ds.drop(['year','month','day','doy'], axis=1) # removing all other time info
ds=ds.set_index(['time'])
ds=ds.drop_duplicates()# just in case

daily={time: dailydata for time,dailydata in ds.groupby(ds.index.date)} # group by daily data and turn to dict
ntime=len(daily) # Useful for later

# load the bathymetry as a proxy for the land
bath=pd.read_csv('../data/bathymetry_EASE.csv', header= None)

datamask= (1-bath.isna()) # make sure to check where we have data
land=(bath.where(bath==3000,np.nan)-3000)

uarr=land.copy().to_numpy().astype(np.float32)
ucube=np.tile(uarr, (ntime,1,1))

vcube=ucube.copy()
interU=ucube.copy()
interV=ucube.copy()

lat_grid = pd.read_csv('../data/latitude_EASE.csv', header=None).values
lon_grid = pd.read_csv('../data/longitude_EASE.csv', header=None).values

y_idx = np.arange(ucube.shape[1])
x_idx = np.arange(ucube.shape[2])


# for i,day in enumerate(daily.keys()):
#     rows = daily[day]['y_EASE'].astype(int).to_numpy()
#     cols = daily[day]['x_EASE'].astype(int).to_numpy()
#     ucube[i,rows,cols]=daily[day]['u_buoy']
#     vcube[i,rows,cols]=daily[day]['v_buoy']

grid_y, grid_x = np.meshgrid(y_idx, x_idx, indexing='ij')  # once, outside the loop

for i, day in enumerate(daily.keys()):
    rows = daily[day]['y_EASE'].astype(int).to_numpy()
    cols = daily[day]['x_EASE'].astype(int).to_numpy()
    ucube[i, rows, cols] = daily[day]['u_buoy']
    vcube[i, rows, cols] = daily[day]['v_buoy']

    valid_u = ~np.isnan(ucube[i])
    valid_v = ~np.isnan(vcube[i])

    interU[i] = griddata(
        (grid_y[valid_u], grid_x[valid_u]),
        ucube[i][valid_u],
        (grid_y, grid_x),
        method='linear'
    )
    interV[i] = griddata(
        (grid_y[valid_v], grid_x[valid_v]),
        vcube[i][valid_v],
        (grid_y, grid_x),
        method='linear'
    )


ds = xr.Dataset(
    {
        'u': (['time', 'y', 'x'], ucube),
        'v': (['time', 'y', 'x'], vcube),
        'u_inter':(['time', 'y', 'x'], interU),
        'v_inter':(['time', 'y', 'x'], interV)
    },
    coords={
        'time': list(daily.keys()),
        'y': y_idx,
        'x': x_idx,
        'lat': (['y', 'x'], lat_grid),
        'lon': (['y', 'x'], lon_grid),
    }
)


# uarr[rows,cols]=day1['u_buoy'].to_numpy()


# plt.imshow(uarr)
# plt.colorbar()
# plt.show()

# print(ucube.shape)
# # for i in day1:

# #     print(i)
# #     umap[day1[datetime(i)]['y_EASE'].astype(int), day1[i]['x_EASE'].astype(int)]=day1[i]['u_buoy']





# plt.imshow(land)
# plt.colorbar()
# plt.show()
# plt.imshow(datamask)
# plt.colorbar()


# xr.DataArray(
#     np.full_like(lat_grid,np.nan),
    
# )

In [ ]:
# 

# inter = ds.interp(x=ds.x, y=ds.y,
#                           method='linear')

inter =griddata((,ds.y), ds.u.isel({'time':0}), (ds.x,ds.y))
plt.imshow(inter.u.isel({'time':0}))


In [ ]:
ds = pd.read_csv('../data/DRIFT_DATA_TRAIN.csv')
lat_grid = pd.read_csv('../data/latitude_EASE.csv', header=None).values
lon_grid = pd.read_csv('../data/longitude_EASE.csv', header=None).values

ds['time']=pd.to_datetime(ds[['year',	'month',	'day']]) # creat a datetime64
ds['lat'] = lat_grid[ds['y_EASE'].astype(int), ds['x_EASE'].astype(int)]
ds['lon'] = lon_grid[ds['y_EASE'].astype(int), ds['x_EASE'].astype(int)]
#ds=ds.set_index(['time'])
ds=ds.drop(['year','month','day','doy','x_EASE','y_EASE'], axis=1)
ds=ds.set_index(['time'])
ds=ds.drop_duplicates()

In [ ]:
daily={time: dailydata for time,dailydata in ds.groupby(ds.index.date)}

In [ ]:
day1=daily[datetime.date(1979,2,19)]

fig, ax = plt.subplots(
    figsize=(8, 8),
    subplot_kw={'projection': ccrs.NorthPolarStereo()}
)
# Set a sensible Arctic extent — adjust bounds to your buoy region
ax.set_extent([-180, 180, 60, 90], crs=ccrs.PlateCarree())

# Add map context

ax.coastlines(alpha=.5)#draw_labels=True, linewidth=0.3, alpha=0.5)
ax.scatter(
     day1.lon,day1.lat,
    transform=ccrs.PlateCarree(),  # tells cartopy the data is in lon/lat, not the plot's projection
)

In [ ]:
df = pd.read_csv('../data/DRIFT_DATA_TRAIN.csv')


yearly = {year: yearlydata.drop(['year'], axis=1) for year,yearlydata in df.groupby(df.year)}

for i in yearly:
    yearly[i]= {doy: dailydata.drop(['month','day','doy'], axis=1) for doy,dailydata in yearly[i].groupby(yearly[i].doy)}

In [ ]:
dff=pd.read_csv('../data/DRIFT_DATA_TRAIN.csv')
dff['time']=pd.to_datetime(dff[['year',	'month',	'day']])
dff=dff.drop(['year','month','day','doy'], axis=1)
dff=dff.set_index(['x_EASE','y_EASE', 'time'])
dsxr=dff.to_xarray()
# xr.DataArray(
#     data=dff.drop(['year','month','day','doy','time','x_EASE','y_EASE'], axis=1),
#     dims=['x','y','time'],
#     coords={
#         'x': dff['x_EASE'],
#         'y': dff['y_EASE'],
#         'time': dff['time'],

#     }
# )
print(dsxr)

In [ ]:
plt.plot(yearly[1979][49].x_EASE,yearly[1979][49].y_EASE,'.')